# Cum am construit `app/app.py` — tutorial Gradio (pas cu pas)

Ideea Gradio, într-o propoziție: **o funcție Python devine o interfață web.**
Tu scrii funcția, Gradio face caseta, butonul și layout-ul.

In [1]:
# o singură dată:  %pip install -q gradio
import gradio as gr
print('Gradio', gr.__version__)

c:\Users\georg\OneDrive\Dokument\Claude\Projects\Cursul Inginerie Ai\echochamber-project-team3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Gradio 6.14.0


## 1. Cel mai simplu Gradio

`gr.Interface` are nevoie de 3 lucruri: `fn` (funcția), `inputs`, `outputs`.

In [2]:
def saluta(nume):
    return 'Salut, ' + nume

gr.Interface(fn=saluta, inputs='text', outputs='text').launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


Atât. Caseta, butonul *Submit*, totul l-a făcut Gradio. Pe asta se construiește
toată aplicația.

## 2. Mai multe input-uri = o listă

Tab-urile noastre au mai multe câmpuri. Dacă funcția are mai multe argumente,
dai o **listă** la `inputs` (ordinea = ordinea argumentelor).

In [3]:
def combina(text, optiune, numar):
    return f'[{optiune} @ {numar}] {text}'

gr.Interface(
    fn=combina,
    inputs=[gr.Textbox(label='Text'),
            gr.Dropdown(['a','b'], value='a', label='Optiune'),
            gr.Slider(1, 10, value=5, label='Numar')],
    outputs=gr.Textbox(label='Rezultat'),
).launch()

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Reține tiparul `[text, dropdown, slider] -> funcție -> text`. **Asta e un tab.**

## 3. Backend fals (ca să rulăm fără chei API)

În `app.py` real sunt 2 importuri din `core/`:

```python
from core.agent import generate_agent_response
from core.graph import run_thread
```

Ca să rulăm tutorialul fără chei, simulăm aceste funcții.

In [4]:
def fake_llm(prompt):
    return '(raspuns simulat) despre: ' + prompt[:70]

def fake_agent(slug, stimulus):
    voci = {
        'anti_sistem':          'Institutiile par din nou rupte de oameni.',
        'pro_european':         'Directia europeana ramane singura ancora solida.',
        'conspirationist':      'Scenariul e clar - ordinul vine de sus.',
        'intelectual_critic':   'Unde sunt dovezile? Analiza lipseste.',
        'personalist_salvator': 'Singurul care poate salva Romania este liderul nostru.',
    }
    return voci.get(slug, f'[{slug}] Raspuns simulat pentru: {stimulus[:40]}')

## 4. Tab-ul Chat

În `app.py`, tab-ul Chat e funcția `chat()` + un `gr.Interface`.

In [5]:
def chat(prompt):
    return fake_llm(prompt) if prompt.strip() else 'Scrie un prompt.'

gr.Interface(
    fn=chat,
    inputs=gr.Textbox(label='Intrebare / prompt', lines=4),
    outputs=gr.Textbox(label='Raspuns', lines=6),
).launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


Tab-ul Chat complet, fără să scriem vreun buton. Gradio l-a făcut.

## 5. Starea partajată: `CFG` și `ART`

Tab-ul **Setări** alege modelul și încarcă o știre. Celelalte tab-uri
trebuie să **vadă** acea știre. Soluția minimă: două dicționare la nivel de modul.

In [6]:
CFG = {'provider': 'gemini', 'model': 'gemini-2.5-flash', 'temp': 0.3}
ART = {'text': '', 'title': ''}

MODEL_CHOICES = [('gemini · gemini-2.5-flash', 'gemini|gemini-2.5-flash'),
                 ('deepseek · deepseek-chat',  'deepseek|deepseek-chat')]

def setup(model_choice, temp, url):
    provider, model = model_choice.split('|')
    CFG['provider'] = provider; CFG['model'] = model; CFG['temp'] = temp
    if url.strip():
        ART['text']  = f'(stire simulata de la {url[:60]})'
        ART['title'] = url[:60]
        return f'Model: {model} | Temp: {temp} | Stire: {ART["title"]}'
    return f'Model: {model} | Temp: {temp} | Fara stire'

## 6. Regula cheie: subiectul intră *peste* știre

- **știre + subiect** → vorbim despre subiect în contextul știrii
- **doar știre** → vorbim despre știre
- **doar subiect** → vorbim direct
- **nimic** → eroare

In [7]:
def _subject(typed):
    typed = (typed or '').strip()
    news  = ART['text'].strip()
    if news and typed:
        return f'{typed}\n\n[In contextul stirii:]\n{news[:600]}'
    if news:  return news[:800]
    if typed: return typed
    return ''

## 7. Tab-ul Agent

Tab-ul Agent = `_subject()` + chemarea backend-ului.

In [8]:
AGENT_SLUGS = ['anti_sistem','pro_european','conspirationist',
               'intelectual_critic','personalist_salvator']

def agent(text, slug):
    s = _subject(text)
    if not s.strip(): return 'Incarca o stire sau scrie un subiect.'
    return fake_agent(slug, s)

gr.Interface(
    fn=agent,
    inputs=[gr.Textbox(label='Subiect (optional)', lines=3),
            gr.Dropdown(AGENT_SLUGS, value=AGENT_SLUGS[0], label='Agent')],
    outputs=gr.Textbox(label='Raspuns agent', lines=6),
).launch()

* Running on local URL:  http://127.0.0.1:7864
* To create a public link, set `share=True` in `launch()`.


Tab-urile **Rezumat**, **Toți agenții** și **Dezbatere** urmează același tipar.

## 8. Punem tab-urile împreună

`gr.Blocks` cu temă + `gr.Tabs()` + `.render()` în fiecare tab.

In [9]:
tab_setup = gr.Interface(setup,
    [gr.Dropdown(MODEL_CHOICES, value=MODEL_CHOICES[0][1], label='Provider · Model'),
     gr.Slider(0, 1, value=0.3, step=0.1, label='Temperatura'),
     gr.Textbox(label='URL stire (optional)')],
    gr.Textbox(label='Configuratie activa'),
)
tab_chat  = gr.Interface(chat,
    gr.Textbox(label='Prompt', lines=4),
    gr.Textbox(label='Raspuns', lines=6))
tab_agent = gr.Interface(agent,
    [gr.Textbox(label='Subiect', lines=3),
     gr.Dropdown(AGENT_SLUGS, value=AGENT_SLUGS[0], label='Agent')],
    gr.Textbox(label='Raspuns agent', lines=6))

with gr.Blocks(theme=gr.themes.Base(), title='EchoChamber Studio') as demo:
    gr.Markdown('# EchoChamber Studio')
    with gr.Tabs():
        with gr.Tab('Setari'):  tab_setup.render()
        with gr.Tab('Chat'):    tab_chat.render()
        with gr.Tab('Agent'):   tab_agent.render()
demo.launch()

C:\Users\georg\AppData\Local\Temp\ipykernel_20824\1276971828.py:15: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Base(), title='EchoChamber Studio') as demo:


* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.


Acesta e scheletul din `app.py`. Funcțiile reale cheamă `core/` în loc de `fake_*`.

## 9. Recapitulare

1. `gr.Interface(fn, inputs, outputs)` — o funcție devine interfață.
2. Mai multe input-uri = o listă. Un astfel de bloc = un tab.
3. `gr.Blocks` + `gr.Tabs` + `.render()` = aplicație cu mai multe tab-uri.
4. Starea partajată = dicționar de modul (`CFG`, `ART`).
5. `_subject()` combină știrea cu inputul utilizatorului.
6. Funcțiile reale înlocuiesc `fake_*`.

---

## Tema 3 — Extensii individuale Gradio

Adaug trei modificări vizibile la mini-aplicația din notebook.

| # | Tip | Ce face |
|---|-----|---------|
| 1 | Tab nou | **Etică** — disclaimer despre natura agenților |
| 2 | Funcție nouă | **Contor text** — numără cuvintele și caracterele |
| 3 | Temă vizuală | `gr.themes.Soft` cu accent albastru |


### Modificare 1 — Tab nou: Etică

In [10]:
ETHICS_MD = (
    '## Ce sunt agentii EchoChamber\n\n'
    'Agentii sunt **roluri discursive simulate**. '
    'Nu sunt persoane reale si nu reprezinta niciun grup social.\n\n'
    '## Ce inseamna outputurile\n\n'
    'Raspunsurile generate **nu sunt afirmatii factuale**. '
    'Contextul RAG este doar material de sprijin.\n\n'
    '## Utilizare responsabila\n\n'
    'Folositi rezultatele pentru educatie si analiza critica.  \n'
    'Nu prezentati outputurile ca opinie publica reala.\n\n'
    '*EchoChamber Studio este un prototip de cercetare si predare.*'
)

gr.Interface(
    fn=lambda: ETHICS_MD,
    inputs=[],
    outputs=gr.Markdown(),
    title='Etica si limite',
).launch()

* Running on local URL:  http://127.0.0.1:7866
* To create a public link, set `share=True` in `launch()`.


### Modificare 2 — Funcție nouă: contor de cuvinte și caractere

In [11]:
def contor_text(text):
    if not text.strip():
        return 'Introdu un text mai intai.'
    cuvinte    = len(text.split())
    caractere  = len(text)
    propozitii = text.count('.') + text.count('!') + text.count('?')
    return (f'Cuvinte: {cuvinte}\n'
            f'Caractere: {caractere}\n'
            f'Propozitii (aprox.): {max(propozitii, 1)}')

gr.Interface(
    fn=contor_text,
    inputs=gr.Textbox(label='Text de analizat', lines=6,
                      placeholder='Lipeste un raspuns generat...'),
    outputs=gr.Textbox(label='Statistici'),
    title='Contor text',
    description='Evalueaza lungimea unui raspuns de agent.',
).launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


### Modificare 3 — Temă vizuală: `gr.themes.Soft` cu accent albastru

In [13]:
with gr.Blocks(theme=gr.themes.Soft(primary_hue='blue'),
               title='EchoChamber Studio') as demo_tema:
    gr.Markdown('# EchoChamber Studio')
    gr.Markdown('*Simulare discurs politic românesc*')
    with gr.Row():
        with gr.Column():
            inp_text  = gr.Textbox(label='Subiect politic',
                                   value='CCR a anulat alegerile.', lines=3)
            inp_agent = gr.Dropdown(AGENT_SLUGS, value='pro_european', label='Agent')
            btn       = gr.Button('Genereaza (simulat)', variant='primary')
        with gr.Column():
            out_resp  = gr.Textbox(label='Raspuns agent', lines=6, interactive=False)
            out_stats = gr.Textbox(label='Statistici', interactive=False)

    def gen_and_count(text, slug):
        resp  = fake_agent(slug, text)
        stats = contor_text(resp)
        return resp, stats

    btn.click(gen_and_count, inputs=[inp_text, inp_agent], outputs=[out_resp, out_stats])

demo_tema.launch()

C:\Users\georg\AppData\Local\Temp\ipykernel_20824\376056419.py:1: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(primary_hue='blue'),


* Running on local URL:  http://127.0.0.1:7869
* To create a public link, set `share=True` in `launch()`.


## Explicatie Tema 3

**Ce am adaugat:**
Am extins mini-aplicatia Gradio cu trei modificari vizibile: un tab de etica,
un contor de text si o tema vizuala personalizata.

**Functie noua creata:**
`contor_text(text)` numara cuvintele, caracterele si propozitiile aproximative
dintr-un raspuns de agent.

**Modificare design / interfata:**
Am aplicat `gr.themes.Soft(primary_hue='blue')` pentru o paleta albastra.
Demo-ul final combina generarea simulata cu contorul intr-un singur `gr.Blocks`.

**Ce as imbunatati:**
As conecta `contor_text` direct la tab-ul Agent din aplicatia reala,
ca statisticile sa apara automat sub fiecare raspuns generat.
